# Website Fingerprinting Attack on JAP Traffic

This notebook demonstrates a complete Website Fingerprinting attack using the JAP dataset.

**What this notebook covers:**
- Loading and exploring the statistical features from the dataset
- Understanding the feature structure (74 statistical features + labels)
- Data preprocessing (scaling, train/test split)
- Training SVM classifiers (Linear, RBF, Polynomial kernels)
- Hyperparameter tuning with GridSearchCV
- Evaluating model performance
- Visualizing results with confusion matrix


**Author:** [Abdulqader Shaawa]
**Paper:** [A Comprehensive Analysis of Website Fingerprinting on the Java Anon Proxy (JAP) using Machine and Deep Learning]
**Dataset:** [https://github.com/kadershawa-hub/WF\JAP\Data]

---

## 1. Setup and installation

In [ ]:
# Install required packages
!pip install scikit-learn pandas matplotlib seaborn gdown

# Import libraries
import os
import gdown
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json
from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_recall_fscore_support, roc_curve, auc
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Set plot style
plt.style.use('ggplot')
sns.set(font_scale=1.2)

print("All libraries imported successfully!")


## 2. Load and Explore Dataset Structure
The dataset is hosted on Google Drive. We'll download it using gdown.

In [ ]:
print("\n" + "="*60)
print("DOWNLOADING YOUR DATASET")
print("="*60)

file_id = '14WrI5_RcbzCJ8terqemjoLs8Wnsjhf43'
output_file = 'filtered_features.csv'

# Create directory
!mkdir -p data

# Download the filtered dataset using gdown
import gdown
url = f'https://drive.google.com/uc?id={file_id}'
gdown.download(url, f'data/{output_file}', quiet=False)

# Set data path
DATA_PATH = f'data/{output_file}'

# Verify
if os.path.exists(DATA_PATH):
    file_size = os.path.getsize(DATA_PATH) / (1024 * 1024)  # MB
    print(f"✅ Dataset downloaded: {DATA_PATH}")
    print(f"📊 File size: {file_size:.2f} MB")

    # Quick preview (removed nrows=3 to load full dataset for processing)
    df = pd.read_csv(DATA_PATH)
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()}")

    # Check for missing values
    missing = df.isnull().sum().sum()
    print(f"  Missing values: {missing}")

    # Check class distribution
    class_counts = df['website'].value_counts()
    print(f"  Number of unique websites: {len(class_counts)}")
    print(f"  Samples per website - Min: {class_counts.min()}, Max: {class_counts.max()}, Mean: {class_counts.mean():.1f}")

    # Display first few rows
    print("\nFirst 5 rows:")
    display(df.head())
print("\n✅ Ready to proceed !")

## STEP 3. CONFIGURE PARAMETERS

In [ ]:
# Data parameters
NUM_CLASSES = 100
RANDOM_STATE = 42
SCALER_TYPE = 'standard'     # Options: 'standard', 'minmax', 'robust'
#NORMALIZE_DATA = True        # Normalize data to [0, 1] range
test_size = 0.2               # Test set size
verbose = 1
# Model parameters
DPI = 600
# Output settings
OUTPUT_DIR = 'output'
# Create output directories if they don't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/results", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/plots", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/models", exist_ok=True)
print(f"✅ Output directory: {OUTPUT_DIR}")

print("✅ Configuration loaded!")
print(f"Using {SCALER_TYPE} scaler")

## STEP 4. Data Preprocessing for Traditional Machine Learning

we need to:
1.   Identify only statistics feature columns
2.   Split into train, validation, and test sets










In [ ]:
# Identify Statistic feature columns (exclude label and sequence features column)

feature_cols = [col for col in df.columns if col != 'website' and pd.api.types.is_numeric_dtype(df[col])]

print(f"\nNumber of statistical features: {len(feature_cols)}")


# Extract features and labels
X = df[feature_cols].values
y = df['website'].values


print(f"feature shape: {X.shape}")

# Initialize scaler
if SCALER_TYPE == 'standard':
    scaler = StandardScaler()
elif SCALER_TYPE == 'minmax':
    scaler = MinMaxScaler()
elif SCALER_TYPE == 'robust':
    scaler = RobustScaler()
else:
    raise ValueError(f"Unknown scaler type: {SCALER_TYPE}")

# Scale features
X_scaled = scaler.fit_transform(X)
print(f"Scaled feature shape: {X_scaled.shape}")

# First split: train (80%) and temp (20%)
X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# Second split: validation (10%) and test (10%) from temp
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## STEP 6. SVM Model Training


In [ ]:
print("\n" + "="*60)
print("STEP 6: SVM Model Training")
print("="*60)

def train_svm(X_train, y_train, X_val, y_val, kernel='rbf', C=1.0, gamma='scale', degree=3, verbose=True):
    """
    Train an SVM classifier.

    Args:
        X_train, y_train: Training data
        X_val, y_val: Validation data
        kernel: SVM kernel ('linear', 'rbf', 'poly')
        C: Regularization parameter
        gamma: Kernel coefficient ('scale', 'auto', or float)
        degree: Degree of the polynomial kernel function ('poly'). Ignored by other kernels.
        verbose: Print progress

    Returns:
        model: Trained SVM model
        train_acc: Training accuracy
        val_acc: Validation accuracy
        training_time: Training time in seconds
    """
    start_time = time.time()

    model = SVC(
        kernel=kernel,
        C=C,
        gamma=gamma,
        degree=degree,  # Pass degree to SVC constructor
        probability=True,  # Enable probability estimates for ROC curves
        random_state=RANDOM_STATE,
        verbose=False
    )

    model.fit(X_train, y_train)
    training_time = time.time() - start_time

    train_acc = model.score(X_train, y_train)
    val_acc = model.score(X_val, y_val)

    if verbose:
        print(f"  Kernel: {kernel}, C={C}, gamma={gamma}")
        if kernel == 'poly': # Only print degree if polynomial kernel
            print(f"  Degree: {degree}")
        print(f"  Training time: {training_time:.2f}s")
        print(f"  Training accuracy: {train_acc:.4f}")
        print(f"  Validation accuracy: {val_acc:.4f}")

    return model, train_acc, val_acc, training_time

def train_linear_svm(X_train, y_train, X_val, y_val, C=1.0, verbose=True):
    """Train a linear SVM (faster for high-dimensional data)."""
    start_time = time.time()

    model = LinearSVC(
        C=C,
        random_state=RANDOM_STATE,
        max_iter=10000,
        dual='auto'
    )

    model.fit(X_train, y_train)
    training_time = time.time() - start_time

    train_acc = model.score(X_train, y_train)
    val_acc = model.score(X_val, y_val)

    if verbose:
        print(f"  Linear SVM, C={C}")
        print(f"  Training time: {training_time:.2f}s")
        print(f"  Training accuracy: {train_acc:.4f}")
        print(f"  Validation accuracy: {val_acc:.4f}")

    return model, train_acc, val_acc, training_time

# Train models
print("\n" + "-"*40)
print("Training SVM Models")
print("-"*40)

models = {}

# Linear SVM
print("\n1. Linear SVM:")
model_linear, train_acc, val_acc, train_time = train_linear_svm(
    X_train, y_train, X_val, y_val, C=1.0
)
models['linear'] = {'model': model_linear, 'train_acc': train_acc, 'val_acc': val_acc, 'time': train_time}

# RBF SVM
print("\n2. RBF SVM (C=1.0, gamma='scale'):")
model_rbf, train_acc, val_acc, train_time = train_svm(
    X_train, y_train, X_val, y_val, kernel='rbf', C=1.0, gamma='scale'
)
models['rbf'] = {'model': model_rbf, 'train_acc': train_acc, 'val_acc': val_acc, 'time': train_time}

# Polynomial SVM
print("\n3. Polynomial SVM (degree=3):")
model_poly, train_acc, val_acc, train_time = train_svm(
    X_train, y_train, X_val, y_val, kernel='poly', C=1.0, gamma='scale', degree=3
)
models['poly'] = {'model': model_poly, 'train_acc': train_acc, 'val_acc': val_acc, 'time': train_time}

## 6.2 Hyperparameter Tuning

In [ ]:
print("\n" + "="*60)
print("STEP 6.2: Hyperparameter Tuning (Grid Search)")
print("="*60)

def grid_search_svm(X_train, y_train, X_val, y_val, param_grid, kernel='rbf', verbose=True):
    """
    Perform grid search for SVM hyperparameters.

    Args:
        X_train, y_train: Training data
        X_val, y_val: Validation data
        param_grid: Dictionary of parameters to search
        kernel: SVM kernel
        verbose: Print progress

    Returns:
        best_model: Best SVM model
        best_params: Best parameters
        cv_results: Cross-validation results
    """
    print(f"\nPerforming grid search for {kernel} SVM...")
    print(f"Parameter grid: {param_grid}")

    start_time = time.time()

    svm = SVC(kernel=kernel, random_state=RANDOM_STATE, probability=True)

    grid_search = GridSearchCV(
        svm,
        param_grid,
        cv=3,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1 if verbose else 0
    )

    grid_search.fit(X_train, y_train)

    search_time = time.time() - start_time

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    best_val_score = grid_search.best_score_

    # Evaluate on validation set
    val_acc = best_model.score(X_val, y_val)

    if verbose:
        print(f"\nSearch completed in {search_time:.2f}s")
        print(f"Best parameters: {best_params}")
        print(f"Best CV score: {best_val_score:.4f}")
        print(f"Validation accuracy: {val_acc:.4f}")

    return best_model, best_params, grid_search.cv_results_

# Define parameter grids
param_grid_rbf = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 0.01, 0.001]
}

param_grid_linear = {
    'C': [0.1, 1, 10, 100]
}

# Perform grid search on filtered dataset (better performance)
print("\nGrid Search on Filtered Dataset (RBF SVM):")
best_rbf, best_rbf_params, rbf_cv_results = grid_search_svm(
    X_train, y_train, X_val, y_val,
    param_grid_rbf, kernel='rbf'
)

print("\nGrid Search on Filtered Dataset (Linear SVM):")
best_linear, best_linear_params, linear_cv_results = grid_search_svm(
    X_train, y_train, X_val, y_val,
    param_grid_linear, kernel='linear'
)

# Update models dictionary with tuned models
models['rbf_tuned'] = {'model': best_rbf, 'params': best_rbf_params, 'val_acc': best_rbf.score(X_val, y_val)}
models['linear_tuned'] = {'model': best_linear, 'params': best_linear_params, 'val_acc': best_linear.score(X_val, y_val)}

# Save best models
import joblib
joblib.dump(best_rbf, os.path.join(OUTPUT_DIR, "models", "best_rbf_svm.pkl"))
joblib.dump(best_linear, os.path.join(OUTPUT_DIR, "models", "best_linear_svm.pkl"))
print("\nBest models saved to disk.")

## 7. Model Evaluation


In [ ]:
print("\n" + "="*60)
print("STEP 7: Model Evaluation on Test Set")
print("="*60)

def evaluate_model(model, X_test, y_test, model_name, label_encoder=None):
    """
    Evaluate a trained model on test data.

    Args:
        model: Trained classifier
        X_test, y_test: Test data
        model_name: Name for output
        label_encoder: Optional label encoder for class names

    Returns:
        accuracy: Test accuracy
        report: Classification report
        y_pred: Predictions
    """
    print(f"\n{'-'*40}")
    print(f"Evaluating: {model_name}")
    print(f"{'-'*40}")

    start_time = time.time()
    y_pred = model.predict(X_test)
    eval_time = time.time() - start_time

    accuracy = accuracy_score(y_test, y_pred)

    # Get class names
    if label_encoder is not None:
        target_names = label_encoder.classes_
    else:
        target_names = np.unique(y_test)

    print(f"Test accuracy: {accuracy:.4f}")
    print(f"Evaluation time: {eval_time:.2f}s")

    # Generate classification report
    report = classification_report(y_test, y_pred, target_names=target_names, output_dict=True)
    print("\nClassification Report (summary):")
    print(f"  Macro avg - Precision: {report['macro avg']['precision']:.4f}")
    print(f"  Macro avg - Recall: {report['macro avg']['recall']:.4f}")
    print(f"  Macro avg - F1-score: {report['macro avg']['f1-score']:.4f}")
    print(f"  Weighted avg - Precision: {report['weighted avg']['precision']:.4f}")
    print(f"  Weighted avg - Recall: {report['weighted avg']['recall']:.4f}")
    print(f"  Weighted avg - F1-score: {report['weighted avg']['f1-score']:.4f}")

    return accuracy, report, y_pred

# Encode labels for class names (use original dataset's label encoder)
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
label_encoder.fit(y_train)

# Evaluate all models on filtered dataset test set
results = []

for model_key, model_info in models.items():
    model = model_info['model']
    if hasattr(model, 'predict'):
        accuracy, report, y_pred = evaluate_model(
            model, X_test, y_test,
            f"Filtered Dataset - {model_key}",
            label_encoder
        )
        results.append({
            'model': model_key,
            'accuracy': accuracy,
            'precision_macro': report['macro avg']['precision'],
            'recall_macro': report['macro avg']['recall'],
            'f1_macro': report['macro avg']['f1-score']
        })


# Create results DataFrame
results_df = pd.DataFrame(results)
print("\n" + "="*40)
print("Summary Results:")
print(results_df.to_string(index=False))

# Save results
results_df.to_csv(os.path.join(OUTPUT_DIR, "results", "svm_results_summary.csv"), index=False)